# Embeddings and a Vector Database with ChromaDB
In this notebook you will:
1. Turn sentences into **embeddings** (vectors of numbers)
2. See that similar meanings give similar vectors
3. Store the vectors in **ChromaDB**, a vector database
4. Run a **similarity search**: find the sentences closest in meaning to a question
5. Add more data and filter results by topic

## 0. Add your API key
Get a free key at https://aistudio.google.com/apikey, then paste it in the cell below.
**Never share your key or upload it to GitHub.**

In [ ]:
# ==========================================
#  >>> PASTE YOUR API KEY BETWEEN THE QUOTES <<<
# ==========================================
API_KEY = "PASTE_YOUR_API_KEY_HERE"

## 1. Install and set up
We need `google-genai` (to make embeddings) and `chromadb` (the vector database). Run the install cell once.

In [ ]:
%pip install -q google-genai chromadb numpy

In [ ]:
import chromadb
import numpy as np
from google import genai
from google.genai import types

if not API_KEY or API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise RuntimeError("Paste your API key into the API_KEY cell above first.")

client = genai.Client(api_key=API_KEY)
EMBED_MODEL = "gemini-embedding-001"
print("Client ready")

## 2. What is an embedding?
An **embedding** turns a piece of text into a list of numbers (a **vector**). Texts with similar *meaning* get vectors that point in similar directions.

Let's embed one sentence and look at it.

In [ ]:
def embed(texts):
    """Turn a list of texts into a list of vectors (768 numbers each)."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    return [e.values for e in result.embeddings]

vec = embed(["The cat sat on the sofa."])[0]
print(f"Numbers in the vector : {len(vec)}")
print(f"First 5 numbers       : {vec[:5]}")

## 3. Closer meaning = closer vectors
**Cosine similarity** measures how close two vectors are: 1.0 means the same direction, and lower means less related. Let's compare a sentence with a similar one and with an unrelated one.

In [ ]:
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

s1, s2, s3 = embed([
    "The cat sat on the sofa.",
    "A kitten is resting on the couch.",
    "The rocket launched into orbit.",
])

print(f"cat vs kitten : {cosine(s1, s2):.3f}   <- similar meaning, no shared key words")
print(f"cat vs rocket : {cosine(s1, s3):.3f}   <- unrelated, so a lower score")

## 4. Our data
Here is a small knowledge base: 9 sentences about 3 topics (animals, food, space). We type it straight into the notebook and embed all of it in one call.

In [ ]:
# (text, topic) pairs - typed in by hand, no files needed
DATA = [
    ("Dogs are loyal pets that love to play fetch.",           "animals"),
    ("A cheetah can run faster than any other land animal.",   "animals"),
    ("Dolphins are intelligent mammals that live in the sea.", "animals"),
    ("Pizza is made with dough, tomato sauce and cheese.",     "food"),
    ("Sushi is a Japanese dish of rice and raw fish.",         "food"),
    ("A bowl of hot soup is perfect on a cold day.",           "food"),
    ("The Moon orbits the Earth about once a month.",          "space"),
    ("Mars is called the red planet because of its rusty soil.", "space"),
    ("Astronauts float because they are in free fall around Earth.", "space"),
]
DOCUMENTS = [text for text, topic in DATA]
TOPICS    = [topic for text, topic in DATA]

doc_vectors = embed(DOCUMENTS)      # one API call for all sentences
print(f"{len(DOCUMENTS)} sentences -> {len(doc_vectors)} vectors of {len(doc_vectors[0])} numbers each")

## 5. Store the vectors in ChromaDB
A **vector database** stores vectors so it can quickly find the ones closest to a query. ChromaDB keeps the **text, the vector and extra info (metadata)** together in a *collection*, a bit like a table.

`chromadb.Client()` runs in memory, so everything disappears when the notebook stops.

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="demo",
    configuration={"hnsw": {"space": "cosine"}},  # measure closeness with cosine
)

collection.add(
    ids=[f"doc{i}" for i in range(len(DOCUMENTS))],
    documents=DOCUMENTS,
    embeddings=doc_vectors,
    metadatas=[{"topic": t} for t in TOPICS],
)
print(f"Stored {collection.count()} items")

## 6. Similarity search
To search, we embed the **question** with the same model, then ask ChromaDB for the `N_RESULTS` closest vectors.

Chroma returns a **distance**: smaller = closer in meaning.

In [ ]:
QUERY = "Which animal is the fastest?"
N_RESULTS = 3

def search(query, n_results=N_RESULTS, where=None):
    q_vec = embed([query])
    res = collection.query(query_embeddings=q_vec, n_results=n_results, where=where)
    print(f"Question: {query}\n")
    for text, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
        print(f"  distance {dist:.3f} | {meta['topic']:7} | {text}")

search(QUERY)

## 7. Search by meaning, not by key words
The questions below share **no words** with the best answer. A normal text search would fail, but the embeddings still match the meaning.

In [ ]:
search("What do people eat from Japan?")
search("Why is there no gravity in a space station?")

## 8. Add more data
A vector database can grow: embed the new sentence and `add` it. Then search again.

In [ ]:
NEW_TEXT = "Penguins cannot fly but they are excellent swimmers."
collection.add(
    ids=["doc_new"],
    documents=[NEW_TEXT],
    embeddings=embed([NEW_TEXT]),
    metadatas=[{"topic": "animals"}],
)
print(f"Now storing {collection.count()} items\n")
search("Which bird swims well?")

## 9. Filter by metadata
Because Chroma stores metadata, you can combine meaning search with a filter. Here we only look inside the `space` topic.

In [ ]:
search("Tell me about something round in the sky", where={"topic": "space"})

## Try it yourself
- Change `QUERY` and `N_RESULTS` in section 6.
- Add 3 sentences of your own (new topic too!) to `DATA` and rerun from section 4.
- Ask a question that matches **nothing** in the data. What does the closest result look like? (Look at the distance.)
- Rewrite the same question in different words and check that the same sentence still comes first.